# 09 NOAA-20 대체 검증: 같은 모델에 다른 VIIRS 위성을 넣으면?

모델은 **Suomi NPP**의 VIIRS로만 학습했습니다. **NOAA-20**(JPSS-1)은 같은 VIIRS 센서를 단 위성으로, FIRMS에 2018년부터 연도별 자료가 있습니다.
Suomi NPP도 2011년에 발사되어 MODIS처럼 오래되었으므로, NPP가 멈춘 뒤에도 **다시 학습하지 않고** 같은 모델로 달력을 이어 갈 수 있는지 확인합니다.

- **학습:** 05와 같은 연도별 시험. 시험 연도 y마다 y−2년까지의 NPP로 학습하고 y−1년으로 조기 종료합니다.
- **시험:** 같은 모델에 y년의 NPP 입력과 NOAA-20 입력을 각각 넣고 실제 MODIS와 비교합니다. NOAA-20은 학습에 한 번도 쓰지 않습니다.
- 위성 여러 대의 탐지 수를 더하지 않습니다. 같은 VIIRS가 같은 땅을 몇십 분 간격으로 지나가서, 더하면 같은 불을 여러 번 셉니다.

NOAA-20 격자 만들기 (MODIS는 2018년부터만 있으면 됩니다):

```bash
python scripts/aggregate.py --modis "data/raw/archive/modis/20{18,19,20,21,22,23,24,25}/*.csv" \
    --viirs "data/raw/archive/viirs_noaa20/**/*.csv" --out data/processed/grid_0.5_month_noaa20.parquet
```

## 미리 정한 기준 (결과를 보기 전에 정함)

"NOAA-20을 NPP 대신 넣어도 된다"고 판정하려면, 2020–2024 시험 5년에서 **셋 다** 만족해야 합니다.

1. NOAA-20 입력의 합계 MAE(10° 블록 × 월) 평균이 NPP 입력의 **1.10배 이하**
2. NOAA-20 입력이 최강 규칙 모델(대륙·위도대·계절, NPP 입력)보다 **5년 중 4년 이상** 오차가 작음
3. 평균 |편향|이 NPP 입력보다 **5%p 이하**로만 큼

화재 활동 지수(10° 지역)의 단계 일치와 이상 F1도 함께 보고하지만 판정 기준에는 넣지 않습니다.

In [ ]:
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate, gbm  # evaluate: 한글 폰트 설정 포함
from fsml.activity import BASELINE_YEARS, agreement, monthly_totals, percentile
from fsml.compare import PRIMARY, _scores as scores
from fsml.export import AQUA_START
from fsml.features import MODIS_RELIABLE_UNTIL, make_features, training_rows
from fsml.paths import EXPORTS, MODELS, grid_path
from fsml.rules import CONTINENT_LEVELS, ModularRuleModel
from fsml.splits import block_id, time_folds

RES = 0.5
TEST_YEARS = [2020, 2021, 2022, 2023, 2024]
RULE = "규칙 (대륙·위도대·계절)"

grid_npp = pd.read_parquet(grid_path(RES, "month"))
grid_n20 = pd.read_parquet(grid_path(RES, "month", "_noaa20"))
npp, FEATURES = make_features(training_rows(grid_npp), RES)
n20, features_n20 = make_features(training_rows(grid_n20), RES)
assert FEATURES == features_n20, "두 격자의 열이 다릅니다 (같은 aggregate.py로 만들었는지 확인)"
yr_npp = pd.to_datetime(npp["period"]).dt.year
yr_n20 = pd.to_datetime(n20["period"]).dt.year
p = pd.to_datetime(grid_n20.loc[grid_n20["viirs_n"].notna(), "period"])
print(f"NPP 학습·시험 행 {len(npp):,} | NOAA-20 행 {len(n20):,} | NOAA-20 기간 {p.min():%Y-%m} … {p.max():%Y-%m}")

## 1. 원래 탐지 수: NOAA-20 ÷ NPP (연도별, 전 세계)

In [ ]:
def yearly(grid, col):
    y = pd.to_datetime(grid["period"]).dt.year
    return grid.groupby(y)[col].sum(min_count=1)

raw = pd.DataFrame({
    "MODIS": yearly(grid_npp, "modis_n"),
    "VIIRS NPP": yearly(grid_npp, "viirs_n"),
    "VIIRS NOAA-20": yearly(grid_n20, "viirs_n"),
}).loc[2018:]
raw["NOAA-20 ÷ NPP"] = raw["VIIRS NOAA-20"] / raw["VIIRS NPP"]
raw.round(3)

## 2. 연도별 시험: 같은 모델, 입력 위성만 바꿈

In [ ]:
for d in (npp, n20):
    d["ai"] = np.nan
    d["rule"] = np.nan
rows = []
t0 = time.time()
for (tr, va, te), y in zip(time_folds(npp, TEST_YEARS), TEST_YEARS):
    model = gbm.fit(npp, FEATURES, tr, va, log_every=0)
    rule = ModularRuleModel(levels=CONTINENT_LEVELS).fit(npp[tr | va])
    for name, d, mask in (("NPP", npp, te), ("NOAA-20", n20, (yr_n20 == y).to_numpy())):
        d.loc[mask, "ai"] = model.predict(d.loc[mask, FEATURES])
        d.loc[mask, "rule"] = rule.predict(d[mask])
        for m, col in (("LightGBM", "ai"), (RULE, "rule")):
            rows.append({"연도": y, "입력": name, "모델": m, **scores(d[mask], d.loc[mask, col].to_numpy(), 10)})
results = pd.DataFrame(rows)
print(f"완료 ({time.time() - t0:.0f}초)")
wide = results.pivot_table(index="연도", columns=["입력", "모델"], values=PRIMARY)
wide.loc["평균"] = wide.mean()
wide.round(1)

In [ ]:
results.groupby(["입력", "모델"])[[PRIMARY, "칸 MAE", "칸 R²(log)", "편향 %"]].mean().round(3)

In [ ]:
ai_npp = wide.loc[TEST_YEARS, ("NPP", "LightGBM")]
ai_n20 = wide.loc[TEST_YEARS, ("NOAA-20", "LightGBM")]
rule_npp = wide.loc[TEST_YEARS, ("NPP", RULE)]
abs_bias = results.assign(b=results["편향 %"].abs()).groupby(["입력", "모델"])["b"].mean()
extra_bias = abs_bias[("NOAA-20", "LightGBM")] - abs_bias[("NPP", "LightGBM")]
checks = {
    "NOAA-20 합계 MAE ≤ NPP × 1.10": ai_n20.mean() <= 1.10 * ai_npp.mean(),
    "NOAA-20이 규칙 모델(NPP)보다 5년 중 4년 이상 우세": int((ai_n20 < rule_npp).sum()) >= 4,
    "평균 |편향| 증가 5%p 이하": extra_bias <= 5,
}
detail = {
    "NOAA-20 ÷ NPP 합계 MAE": f"{ai_n20.mean() / ai_npp.mean():.3f}",
    "NOAA-20이 규칙보다 나은 해": f"{int((ai_n20 < rule_npp).sum())} / {len(TEST_YEARS)}",
    "평균 |편향| (NPP → NOAA-20)": f"{abs_bias[('NPP', 'LightGBM')]:.1f}% → {abs_bias[('NOAA-20', 'LightGBM')]:.1f}%",
    **{f"기준: {k}": "통과" if v else "미달" for k, v in checks.items()},
}
print("판정:", "NOAA-20 대체 가능 (기준 3개 통과)" if all(checks.values()) else
      f"NOAA-20 대체 보류 (기준 {sum(not v for v in checks.values())}개 미달)")
pd.Series(detail)

In [ ]:
ax = wide.loc[TEST_YEARS, [("NPP", "LightGBM"), ("NOAA-20", "LightGBM"), ("NPP", RULE)]].plot.bar(
    figsize=(9, 4), rot=0, color=["#2a78d6", "#eb6834", "#1baf7a"], edgecolor="#fcfcfb", linewidth=2)
ax.legend(["AI · NPP 입력", "AI · NOAA-20 입력 (학습에 안 씀)", "규칙 모델 · NPP 입력"], frameon=False)
ax.set_xlabel("시험 연도")
ax.set_ylabel("합계 MAE (10° 블록 × 월, 낮을수록 좋음)")
ax.set_title("같은 모델에 다른 VIIRS 위성을 넣었을 때")
ax.grid(axis="y", color="#e4e3df", lw=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
(EXPORTS / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(EXPORTS / "figures" / "09_noaa20_transfer.png", dpi=160)

## 3. 화재 활동 지수 (10° 지역, 시험 5년)

In [ ]:
truth_rows = grid_npp[grid_npp["modis_n"].notna() & pd.to_datetime(grid_npp["period"]).between(AQUA_START, MODIS_RELIABLE_UNTIL)]
with_region = lambda d: d.assign(region=block_id(d, 10))
t = monthly_totals(with_region(truth_rows), ["modis_n"], "region")
for name, d in (("npp", npp), ("n20", n20)):
    part = monthly_totals(with_region(d[d["ai"].notna()]), ["ai"], "region").rename(columns={"ai": name})
    t = t.merge(part, on=["region", "period"], how="left")
    inside = t["period"].between(part["period"].min(), part["period"].max())
    t.loc[inside, name] = t.loc[inside, name].fillna(0)  # 그 기간에 행이 없는 지역 = 탐지 0
t["modis_pct"] = percentile(t, "modis_n", "region")
for name in ("npp", "n20"):
    t[f"{name}_pct"] = percentile(t, name, "region", base_col="modis_n")
month = t["period"].dt.month
base = t[t["period"].dt.year.between(*BASELINE_YEARS)]
ever = base.groupby(["region", base["period"].dt.month])["modis_n"].max() > 0
active = pd.MultiIndex.from_arrays([t["region"], month]).map(ever).fillna(False).to_numpy(bool)
ev = active & t["period"].dt.year.isin(TEST_YEARS).to_numpy() & t[["npp_pct", "n20_pct"]].notna().all(axis=1).to_numpy()
pd.DataFrame({
    "AI · NPP 입력": agreement(t.loc[ev, "modis_pct"], t.loc[ev, "npp_pct"]),
    "AI · NOAA-20 입력": agreement(t.loc[ev, "modis_pct"], t.loc[ev, "n20_pct"]),
}).round(3)

## 4. 2025년: 최종 모델에 두 위성을 넣은 값이 서로 맞나

2025년은 Terra 궤도 변화로 MODIS를 정답으로 쓰지 않는 해입니다(07). 달력은 이 해를 VIIRS 환산값으로 채우므로,
NPP 대신 NOAA-20으로 채웠을 때 값이 얼마나 달라지는지 봅니다. 최종 모델(`models/lgbm_harmonize.txt`, 2012–2024 NPP로 학습)을 씁니다.

In [ ]:
final = lgb.Booster(model_file=str(MODELS / "lgbm_harmonize.txt"))
assert final.feature_name() == FEATURES


def predict_year(grid, y):
    rows = grid[(pd.to_datetime(grid["period"]).dt.year == y) & grid["viirs_n"].notna()]
    f, _ = make_features(rows, RES)
    return f.assign(pred=final.predict(f[FEATURES]), region=block_id(f, 10))


e_npp, e_n20 = predict_year(grid_npp, 2025), predict_year(grid_n20, 2025)
monthly = pd.DataFrame({
    "NPP → MODIS 기준": e_npp.groupby("period")["pred"].sum(),
    "NOAA-20 → MODIS 기준": e_n20.groupby("period")["pred"].sum(),
})
monthly["NOAA-20 ÷ NPP"] = monthly.iloc[:, 1] / monthly.iloc[:, 0]
h25 = pd.read_parquet(EXPORTS / "harmonized.parquet", columns=["period", "harmonized", "source"])
h25 = h25[h25["source"] == "viirs_model"].groupby("period")["harmonized"].sum()
print("확인: NPP 값 = 결과 파일의 2025년 값", np.allclose(monthly.iloc[:, 0].to_numpy(), h25.reindex(monthly.index).to_numpy(), rtol=1e-6))
blocks = pd.concat({"npp": e_npp.groupby(["region", "period"])["pred"].sum(),
                    "n20": e_n20.groupby(["region", "period"])["pred"].sum()}, axis=1).fillna(0)
print(f"10° 블록 × 월 상관 {np.corrcoef(blocks['npp'], blocks['n20'])[0, 1]:.3f}, "
      f"합계 차이 {blocks['n20'].sum() / blocks['npp'].sum() - 1:+.1%}")
monthly.round(2)

## 판정과 해석

**판정: NOAA-20 대체 가능.** 미리 정한 기준 3개를 모두 통과했습니다.

| 시험 2020–2024 (평균) | 합계 MAE | 편향 |
| --- | --- | --- |
| AI · NPP 입력 | 149.6 | −0.3% |
| **AI · NOAA-20 입력 (학습에 안 씀)** | **118.9** | +1.4% |
| 규칙 (대륙·위도대·계절) · NPP 입력 | 221.1 | +0.5% |

- NOAA-20 입력은 5년 모두 규칙 모델보다 나았고, 평균 오차는 NPP 입력의 0.80배였습니다.
- NOAA-20이 더 좋게 나온 것은 2022·2024년 때문입니다. 이 두 해에는 NPP 탐지 수가 MODIS의 평소 4.4배에서 2.4~3.2배로 떨어진 달이 있습니다(2022-07·08, 2024-07, 2024-11). NPP 자료가 빠진 것으로 보이고, 그 달에도 NOAA-20은 정상이었습니다. 나머지 해의 차이는 2020 −6%, 2021 −3%, 2023 +11%입니다.
- 화재 활동 지수(10° 지역, 시험 5년): 단계 일치 67.1% → 67.4%, 이상 F1 0.669 → 0.684.
- 2025년(최종 모델): 두 입력으로 만든 값의 10° 블록 × 월 상관은 0.999, 연 합계 차이는 −0.2%, 월별 차이는 −2% ~ +6%입니다.
- 원래 탐지 수 NOAA-20 ÷ NPP는 2019–2025년 0.98–1.07입니다(2018년은 NOAA-20 자료가 4월부터라 0.76).

**뜻:** 다시 학습하지 않아도 NOAA-20을 넣어 같은 품질의 MODIS 기준 값을 만들 수 있습니다. NPP가 멈추거나 자료가 빠진 달은 NOAA-20으로 대신합니다. 이때도 위성 여러 대의 탐지 수를 더하지는 않습니다.